# 프로젝트 : 뉴스 기사 요약해보기


### 준비. 라이브러리 설치 및 임포트

In [1]:
!pip install --upgrade summa   # Step 5의 추출적 요약(원문 문장을 그대로 뽑는 방식)에 쓸 라이브러리
!pip install --upgrade nltk    # 불용어(stopwords) 목록을 가져오기 위한 라이브러리

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for summa: filename=summa-1.2.0-py3-none-any.whl size=54387 sha256=b7633d34a97cea4c34f9a0ffc964c78ad2019f9d5eb4dde992f354df0dd72dce
  Stored in directory: /root/.cache/pip/wheels/75/be/7d/71f9113116fbdaa9ae7287a3017e9f1c6fbe95cadf170edbbd
Successfully built summa
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.2 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1


In [13]:
import os                                   # 파일 경로/환경 관련 기본 기능 (여기선 거의 안 쓰지만 관례상 import)
import re                                   # 정규표현식 — 전처리에서 특수문자/HTML 잔재 등을 패턴으로 지우는 데 사용
import numpy as np                          # 배열 연산, 셔플/슬라이싱 등 수치 계산용
import pandas as pd                         # CSV를 표(DataFrame) 형태로 다루기 위한 라이브러리
import matplotlib.pyplot as plt             # 길이 분포, loss 그래프 등을 그리기 위한 시각화 라이브러리
%matplotlib inline
from collections import Counter             # 단어 등장 횟수를 세는 데 사용 (단어 집합 만들 때 필요)
from bs4 import BeautifulSoup               # HTML 태그(<br/> 등) 제거용
import nltk

# NLTK stopwords 다운로드 시 프록시 관련 보안 오류를 해결하기 위해 환경 변수 설정
os.environ['NLTK_ALLOW_PROXIED_URLOPEN'] = '1'

# NLTK 데이터 경로를 명시적으로 설정하여 stopwords 리소스 탐색 오류 방지
nltk.data.path = ['/root/nltk_data'] + nltk.data.path

nltk.download('stopwords')                  # 영어 불용어(the, a, is 같은 의미 적은 단어) 목록 다운로드
from nltk.corpus import stopwords

import torch                                # PyTorch 본체 — 텐서 연산과 GPU 연산의 기반
import torch.nn as nn                       # 신경망 층(Embedding, LSTM, Linear 등)을 만드는 모듈
import torch.nn.functional as F             # softmax처럼 학습 파라미터가 없는 함수형 연산 모음
import torch.optim as optim                 # AdamW 같은 옵티마이저(가중치 업데이트 규칙) 모음
from torch.utils.data import DataLoader, TensorDataset  # 데이터를 배치 단위로 잘라서 공급해주는 도구
from torch.nn.utils.rnn import pad_sequence # 길이가 제각각인 문장들을 한 텐서로 묶기 위해 길이를 맞춰주는 함수

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='bs4')  # BeautifulSoup의 경고 메시지(파서 관련)를 숨김

print("임포트 완료")

임포트 완료


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Step 1. 데이터 수집하기

데이터는 `news_summary_more.csv`(뉴스 헤드라인/본문)를 사용한다. `headlines`가 강의의 `Summary` 역할(짧은 정답 요약), `text`가 강의의 `Text` 역할(원문)이다.

In [3]:
import urllib.request
# urlretrieve: 인터넷의 파일(csv)을 로컬(코랩 환경)에 그대로 내려받는 함수
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/sunnysai12345/News_Summary/master/news_summary_more.csv",
    filename="news_summary_more.csv"
)
# encoding='iso-8859-1' (Latin-1): 이 csv가 UTF-8이 아닌 다른 방식으로 저장돼 있어서,
# 기본값(UTF-8)으로 읽으면 특수문자가 깨지거나 에러가 날 수 있어 명시적으로 지정
data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')
data = data[['headlines', 'text']]  # 필요한 두 열만 남김 (headlines=정답 요약, text=원문)
print('전체 샘플수 :', len(data))
data.sample(10)  # 무작위로 10개 뽑아서 데이터 형태를 눈으로 확인

전체 샘플수 : 98401


,headlines,text
6060,Get the f**k out: Santa to kids as fire alarm ...,A man dressed as Santa Claus at an event in th...
74171,Lebanese Army raises Spanish flag for Barcelon...,The Lebanese Army raised a Spanish flag alongs...
91548,Immature Kejriwal's blame-game irked people: B...,"Advocate Prashant Bhushan, who co-founded the ..."
95591,Trump declares April National Sexual Assault A...,US President Donald Trump has declared April t...
76118,Anti-tourism marches spreading across Europe,Anti-tourism marches are spreading across Euro...
69664,10 stray dogs maul minor boy to death in Mumbai,A Class 2 student died after being bitten by a...
39095,2 men lynched after being mistaken as kidnappe...,Two men were lynched in Assam's Karbi Anglong ...
17125,Women made to remove mangalsutras at exam cent...,A few married women taking a Telangana State P...
10540,10-year-old deaf and mute girl dies after bein...,A 10-year-old deaf and mute girl died days aft...
67540,"3,000 killed in Syria in the deadliest month o...","At least 3,000 people including 955 civilians ..."


Step 5(Summa 추출적 요약)에서는 정제되지 않은 원문이 필요하다. 정제되기 전에 원본 `text`를 별도 열로 미리 보존해둔다.

In [4]:
data['text_raw'] = data['text']  # Step 5에서 다시 쓸 정제 전 원문
print('=3')

=3


## Step 2. 데이터 전처리하기 (추상적 요약)

강의(9/4)의 전처리 파이프라인을 그대로 가져오되, 아래 세 가지는 이 데이터셋에 맞게 다시 판단했다.
- **불용어 제거 여부**: `headlines`를 몇 개 직접 확인해보니 전치사·관사가 살아있는 완결된 문장 구조였다 (예: "Malaysia bans travel to North Korea over escalating tensions"). 강의의 `Summary`와 같은 이유로 **불용어를 제거하지 않는다**.
- **최대 길이(`text_max_len`, `summary_max_len`)**: 이 데이터의 실제 길이 분포를 다시 확인하고 정한다 (아래에서 직접 확인).
- **단어 집합 크기(`src_vocab_size`, `tar_vocab_size`)**: 이 데이터의 희귀 단어 비율을 다시 확인하고 정한다 (아래에서 직접 확인).

### 중복 샘플과 NULL 값이 존재하는 샘플 제거

In [5]:
# nunique(): 중복을 뺀 "고유한" 값의 개수. 전체 샘플 수와 차이가 나면 중복 기사가 섞여 있다는 뜻
print('text 열에서 중복을 배제한 유일한 샘플의 수 :', data['text'].nunique())
print('headlines 열에서 중복을 배제한 유일한 샘플의 수 :', data['headlines'].nunique())

text 열에서 중복을 배제한 유일한 샘플의 수 : 98360
headlines 열에서 중복을 배제한 유일한 샘플의 수 : 98280


In [6]:
# inplace=True 를 설정하면 DataFrame 타입 값을 return 하지 않고 data 내부를 직접적으로 바꿉니다
data.drop_duplicates(subset=['text'], inplace=True)
print('전체 샘플수 :', len(data))

전체 샘플수 : 98360


In [7]:
print(data.isnull().sum())  # 각 열마다 비어있는(NULL/NaN) 값이 몇 개인지 확인

headlines    0
text         0
text_raw     0
dtype: int64


In [8]:
data.dropna(axis=0, inplace=True)  # NULL 값이 하나라도 있는 행(샘플) 전체를 삭제
print('전체 샘플수 :', len(data))

전체 샘플수 : 98360


### 텍스트 정규화와 불용어 제거

In [9]:
contractions = {"ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have", "couldn't": "could not",
                           "didn't": "did not",  "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not", "haven't": "have not",
                           "he'd": "he would","he'll": "he will", "he's": "he is", "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
                           "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have","I'm": "I am", "I've": "I have", "i'd": "i would",
                           "i'd've": "i would have", "i'll": "i will",  "i'll've": "i will have","i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would",
                           "it'd've": "it would have", "it'll": "it will", "it'll've": "it will have","it's": "it is", "let's": "let us", "ma'am": "madam",
                           "mayn't": "may not", "might've": "might have","mightn't": "might not","mightn't've": "might not have", "must've": "must have",
                           "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have","o'clock": "of the clock",
                           "oughtn't": "ought not", "oughtn't've": "ought not have", "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have",
                           "she'd": "she would", "she'd've": "she would have", "she'll": "she will", "she'll've": "she will have", "she's": "she is",
                           "should've": "should have", "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have","so's": "so as",
                           "this's": "this is","that'd": "that would", "that'd've": "that would have", "that's": "that is", "there'd": "there would",
                           "there'd've": "there would have", "there's": "there is", "here's": "here is","they'd": "they would", "they'd've": "they would have",
                           "they'll": "they will", "they'll've": "they will have", "they're": "they are", "they've": "they have", "to've": "to have",
                           "wasn't": "was not", "we'd": "we would", "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have", "we're": "we are",
                           "we've": "we have", "weren't": "were not", "what'll": "what will", "what'll've": "what will have", "what're": "what are",
                           "what's": "what is", "what've": "what have", "when's": "when is", "when've": "when have", "where'd": "where did", "where's": "where is",
                           "where've": "where have", "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have",
                           "why's": "why is", "why've": "why have", "will've": "will have", "won't": "will not", "won't've": "will not have",
                           "would've": "would have", "wouldn't": "would not", "wouldn't've": "would not have", "y'all": "you all",
                           "y'all'd": "you all would","y'all'd've": "you all would have","y'all're": "you all are","y'all've": "you all have",
                           "you'd": "you would", "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have",
                           "you're": "you are", "you've": "you have"}

print("정규화 사전의 수: ", len(contractions))

정규화 사전의 수:  120


In [10]:
# 데이터 전처리 함수 (9/4 강의와 동일)
def preprocess_sentence(sentence, remove_stopwords=True):
    sentence = sentence.lower() # 텍스트 소문자화
    sentence = BeautifulSoup(sentence, "lxml").text # <br />, <a href = ...> 등의 html 태그 제거
    sentence = re.sub(r'\([^)]*\)', '', sentence) # 괄호로 닫힌 문자열 (...) 제거
    sentence = re.sub('"','', sentence) # 쌍따옴표 제거
    sentence = ' '.join([contractions[t] if t in contractions else t for t in sentence.split(" ")]) # 약어 정규화
    sentence = re.sub(r"'s\b","", sentence) # 소유격 제거
    sentence = re.sub("[^a-zA-Z]", " ", sentence) # 영어 외 문자는 공백으로 변환
    sentence = re.sub('[m]{2,}', 'mm', sentence) # 반복 글자 정리

    if remove_stopwords:
        tokens = ' '.join(word for word in sentence.split() if not word in stopwords.words('english') if len(word) > 1)
    else:
        tokens = ' '.join(word for word in sentence.split() if len(word) > 1)
    return tokens
print('=3')

=3


In [14]:
# 함수가 잘 동작하는지 실제 샘플로 확인
print("전처리 전 text :", data['text'].iloc[0])
print("전처리 후 text :", preprocess_sentence(data['text'].iloc[0]))
print("전처리 전 headlines :", data['headlines'].iloc[0])
print("전처리 후 headlines :", preprocess_sentence(data['headlines'].iloc[0], False))

전처리 전 text : Saurav Kant, an alumnus of upGrad and IIIT-B's PG Program in Machine learning and Artificial Intelligence, was a Sr Systems Engineer at Infosys with almost 5 years of work experience. The program and upGrad's 360-degree career support helped him transition to a Data Scientist at Tech Mahindra with 90% salary hike. upGrad's Online Power Learning has powered 3 lakh+ careers.
전처리 후 text : saurav kant alumnus upgrad iiit pg program machine learning artificial intelligence sr systems engineer infosys almost years work experience program upgrad degree career support helped transition data scientist tech mahindra salary hike upgrad online power learning powered lakh careers
전처리 전 headlines : upGrad learner switches to career in ML & Al with 90% salary hike
전처리 후 headlines : upgrad learner switches to career in ml al with salary hike


In [ ]:
# 전체 text 데이터에 대한 전처리
clean_text = []
for sentence in data['text']:
    clean_text.append(preprocess_sentence(sentence, remove_stopwords=True))

# headlines에는 불용어 제거 X (전치사/관사가 있는 완결된 문장 구조라 판단 — 위 마크다운 참고)
clean_headlines = []
for sentence in data['headlines']:
    clean_headlines.append(preprocess_sentence(sentence, remove_stopwords=False))

print("Text 전처리 후 결과: ", clean_text[:3])
print("headlines 전처리 후 결과: ", clean_headlines[:3])

In [ ]:
data['text'] = clean_text
data['headlines'] = clean_headlines

# 전처리 과정(불용어 제거 등)에서 모든 단어가 사라져 빈 문자열('')이 된 샘플이 있을 수 있음
# 이런 빈 문자열은 isnull()로는 안 잡히므로, 먼저 NaN으로 바꿔줘야 dropna로 걸러낼 수 있음
data.replace('', np.nan, inplace=True)
print(data.isnull().sum())

In [ ]:
data.dropna(axis=0, inplace=True)  # 위에서 NaN으로 바꾼, 즉 전처리 후 내용이 남지 않은 샘플 제거
print('전체 샘플수 :', len(data))

### 샘플의 최대 길이 정하기

In [ ]:
# 길이 분포 출력
import matplotlib.pyplot as plt
%matplotlib inline

text_len = [len(s.split()) for s in data['text']]
summary_len = [len(s.split()) for s in data['headlines']]

print('텍스트의 최소 길이 : {}'.format(np.min(text_len)))
print('텍스트의 최대 길이 : {}'.format(np.max(text_len)))
print('텍스트의 평균 길이 : {}'.format(np.mean(text_len)))
print('요약의 최소 길이 : {}'.format(np.min(summary_len)))
print('요약의 최대 길이 : {}'.format(np.max(summary_len)))
print('요약의 평균 길이 : {}'.format(np.mean(summary_len)))

plt.figure()
plt.subplot(1,2,1)
plt.boxplot(text_len)
plt.title('Text')
plt.subplot(1,2,2)
plt.boxplot(summary_len)
plt.title('Summary')
plt.tight_layout()
plt.show()

plt.figure()
plt.title('Text')
plt.hist(text_len, bins = 40)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

plt.figure()
plt.title('Summary')
plt.hist(summary_len, bins = 40)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
text_max_len = 45      # 위 분포 확인 결과를 바탕으로 결정 (아래 below_threshold_len으로 커버율 확인)
summary_max_len = 13
print('=3')

In [ ]:
# max_len을 하나 정했을 때, "그 길이 이하인 샘플이 전체의 몇 %인가"를 계산하는 함수
# → 이 비율(커버율)이 높아야 max_len으로 자르고 남는 데이터가 대부분 살아남는다는 뜻
def below_threshold_len(max_len, nested_list):
  cnt = 0
  for s in nested_list:
    if(len(s.split()) <= max_len):
        cnt = cnt + 1
  print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s'%(max_len, (cnt / len(nested_list))))
print('=3')

In [ ]:
below_threshold_len(text_max_len, data['text'])
below_threshold_len(summary_max_len, data['headlines'])

In [ ]:
# 위에서 정한 max_len을 넘는(=너무 긴) 샘플은 학습 데이터에서 제외
data = data[data['text'].apply(lambda x: len(x.split()) <= text_max_len)]
data = data[data['headlines'].apply(lambda x: len(x.split()) <= summary_max_len)]

print('전체 샘플수 :', len(data))

### 시작 토큰과 종료 토큰 추가하기

In [ ]:
# 요약 데이터에는 시작 토큰과 종료 토큰을 추가한다.
data['decoder_input'] = data['headlines'].apply(lambda x : 'sostoken '+ x)
data['decoder_target'] = data['headlines'].apply(lambda x : x + ' eostoken')
data.head()

### 훈련데이터와 테스트데이터 나누기

`raw_text`(Step 5용 원본)도 나머지 배열과 똑같은 순서로 섞고 똑같은 비율로 잘라야, 나중에 "같은 기사"끼리 비교할 수 있다.

In [ ]:
encoder_input = np.array(data['text']) # 인코더의 입력
decoder_input = np.array(data['decoder_input']) # 디코더의 입력
decoder_target = np.array(data['decoder_target']) # 디코더의 레이블
raw_text = np.array(data['text_raw']) # Step 5(Summa)용 원본 — 반드시 위 세 배열과 같이 섞고 같이 잘라야 함
print('=3')

In [ ]:
# 배열 하나를 직접 섞지 않고, 대신 "순서(인덱스)"를 섞는다.
# 이렇게 하면 아래에서 encoder_input/decoder_input/decoder_target/raw_text 네 배열 모두에
# 똑같은 indices를 적용해서, "같은 기사"끼리 짝이 흐트러지지 않게 섞을 수 있다.
indices = np.arange(encoder_input.shape[0])
np.random.shuffle(indices)
print(indices)

In [ ]:
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]
raw_text = raw_text[indices]
print('=3')

In [ ]:
n_of_val = int(len(encoder_input)*0.2)  # 전체의 20%를 테스트(검증)용으로 떼어놓음
print('테스트 데이터의 수 :', n_of_val)

In [ ]:
# 음수 인덱스 슬라이싱: [:-n_of_val]은 "뒤에서 n_of_val개를 뺀 나머지 전체(앞부분)",
# [-n_of_val:]은 "뒤에서 n_of_val개"를 뜻한다. 즉 뒤쪽 20%를 테스트셋으로, 나머지 80%를 훈련셋으로 나눈다.
encoder_input_train = encoder_input[:-n_of_val]
decoder_input_train = decoder_input[:-n_of_val]
decoder_target_train = decoder_target[:-n_of_val]
raw_text_train = raw_text[:-n_of_val]

encoder_input_test = encoder_input[-n_of_val:]
decoder_input_test = decoder_input[-n_of_val:]
decoder_target_test = decoder_target[-n_of_val:]
raw_text_test = raw_text[-n_of_val:]

print('훈련 데이터의 개수 :', len(encoder_input_train))
print('훈련 레이블의 개수 :', len(decoder_input_train))
print('테스트 데이터의 개수 :', len(encoder_input_test))
print('테스트 레이블의 개수 :', len(decoder_input_test))

### 단어 집합(vocabulary) 만들기 및 정수 인코딩

In [ ]:
def src_tokenizer(text): # 토크나이저 정의 — 문장을 "단어 리스트"로 쪼개는 함수
    text = text.lower()                          # 대소문자 통일 (Dog와 dog를 같은 단어로 취급)
    text = re.sub(r"[^a-zA-Z0-9]+", " ", text)    # 알파벳/숫자가 아닌 문자는 전부 공백으로 (혹시 남은 특수문자 방지용)
    return text.split()                           # 공백 기준으로 쪼개서 단어 리스트 반환

def build_vocab(texts):
    vocab = {"<PAD>": 0, "<UNK>": 1}  # 0번=패딩(빈 자리 채움용), 1번=미등록 단어(Out-Of-Vocabulary)용으로 예약
    word_counter = Counter()          # 단어별 등장 횟수를 세는 카운터
    for text in texts:
        word_counter.update(src_tokenizer(text))
    for word, _ in word_counter.most_common():  # 등장 빈도가 높은 단어부터 순서대로
        if word not in vocab:
            vocab[word] = len(vocab)  # 새 단어에 다음 정수 번호를 부여 (2, 3, 4, ...)
    return vocab

src_vocab = build_vocab(encoder_input_train)  # 일단 전체 단어에 번호를 매겨봄 (아래에서 희귀 단어 비율을 보고 크기를 제한할 예정)
print('=3')

In [ ]:
# 단어 집합을 전부 다 쓰면 크기가 너무 커진다. 등장 횟수가 threshold보다 적은 "희귀 단어"가
# 전체에서 얼마나 되는지 미리 확인해서, 나중에 vocab_size(단어 집합 크기)를 얼마로 자를지 판단하는 근거로 쓴다.
threshold = 10

word_counter = Counter()
for text in encoder_input_train:
    word_counter.update(src_tokenizer(text))

total_cnt = len(word_counter)   # 전체 고유 단어 수
total_freq = sum(word_counter.values())  # 전체 단어 등장 횟수 총합
rare_cnt = sum(1 for count in word_counter.values() if count < threshold)   # threshold 미만으로 등장한 단어 개수
rare_freq = sum(count for count in word_counter.values() if count < threshold)  # 그 희귀 단어들의 등장 횟수 총합

print('단어 집합(vocabulary)의 크기 :', total_cnt)
print('등장 빈도가 %s번 미만인 단어의 수: %s'%(threshold, rare_cnt))
print('단어 집합에서 희귀 단어를 제외시킬 경우의 단어 집합의 크기 %s'%(total_cnt - rare_cnt))
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)

In [ ]:
# build_vocab과 거의 같지만, most_common(vocab_size - 2)로 "빈도 상위 vocab_size-2개"만 등록한다.
# -2는 위에서 미리 예약해둔 <PAD>, <UNK> 두 자리를 빼기 위함 (합쳐서 정확히 vocab_size가 되도록)
def build_limited_vocab(texts, vocab_size):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    word_counter = Counter()
    for text in texts:
        word_counter.update(src_tokenizer(text))
    for word, _ in word_counter.most_common(vocab_size - 2):
        vocab[word] = len(vocab)
    return vocab

src_vocab_size = 20000  # 위 threshold 분석 결과(등장 10회 이상 단어 수)를 참고해 결정
src_vocab = build_limited_vocab(encoder_input_train, src_vocab_size)
print('=3')

In [ ]:
# 단어(문자열)로 이루어진 문장을, 모델이 계산할 수 있는 "정수 번호의 나열"로 바꾸는 함수
def text_to_sequence(texts, vocab):
    sequences = []
    for text in texts:
        # vocab.get(word, vocab["<UNK>"]): 단어장에 있으면 그 번호, 없으면(=20000개 안에 못 든 단어) UNK 번호로 대체
        sequence = [vocab.get(word, vocab["<UNK>"]) for word in src_tokenizer(text)]
        sequences.append(sequence)
    return sequences

encoder_input_train_seq = text_to_sequence(encoder_input_train, src_vocab)
encoder_input_test_seq = text_to_sequence(encoder_input_test, src_vocab)

print(encoder_input_train_seq[:2])

In [ ]:
# 요약(headlines)용 토크나이저 — src_tokenizer와 로직은 동일하지만, 인코더/디코더 단어장을 분리해서
# 관리하기 위해 이름만 따로 둔다 (원문 어휘와 요약 어휘의 등장 빈도 분포가 다르기 때문)
def tar_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9]+", " ", text)
    return text.split()

tar_vocab = build_vocab(decoder_input_train)
print('=3')

In [ ]:
# 요약 쪽 단어장도 마찬가지로 희귀 단어 비율을 확인 (다만 요약 문장은 짧고 어휘가 더 한정적이라
# threshold를 text 쪽(10)보다 낮은 3으로 잡아본다)
threshold = 3

word_counter = Counter()
for text in decoder_input_train:
    word_counter.update(tar_tokenizer(text))

total_cnt = len(word_counter)
total_freq = sum(word_counter.values())
rare_cnt = sum(1 for count in word_counter.values() if count < threshold)
rare_freq = sum(count for count in word_counter.values() if count < threshold)

print('단어 집합(vocabulary)의 크기 :', total_cnt)
print('등장 빈도가 %s번 미만인 단어의 수: %s'%(threshold, rare_cnt))
print('단어 집합에서 희귀 단어를 제외시킬 경우의 단어 집합의 크기 %s'%(total_cnt - rare_cnt))
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)

In [ ]:
tar_vocab_size = 15000  # 위 threshold 분석 결과를 참고해 결정
# decoder_input(sostoken+요약)과 decoder_target(요약+eostoken)은 같은 단어들로 이루어져 있으므로
# list(...) + list(...)로 두 리스트를 하나로 합쳐서(=단어장 통계를 함께 계산해서) 단어장을 만든다
# (numpy 문자열 배열끼리 그냥 +로 더하면 리스트 합치기가 아니라 원소별 문자열 이어붙이기가 되므로, list()로 먼저 바꿔서 더함)
tar_vocab = build_limited_vocab(list(decoder_input_train) + list(decoder_target_train), tar_vocab_size)

decoder_input_train_seq = text_to_sequence(decoder_input_train, tar_vocab)
decoder_target_train_seq = text_to_sequence(decoder_target_train, tar_vocab)
decoder_input_test_seq = text_to_sequence(decoder_input_test, tar_vocab)
decoder_target_test_seq = text_to_sequence(decoder_target_test, tar_vocab)

print('input ', decoder_input_train_seq[:2])
print('target ', decoder_target_train_seq[:2])

In [ ]:
# decoder_input의 길이가 1이라는 건 sostoken만 남고 실제 요약 단어가 다 사라졌다는 뜻
# (전처리 중 단어가 전부 걸러진 경우) — 이런 샘플은 학습에 의미가 없으므로 제거한다
drop_train = [index for index, sentence in enumerate(decoder_input_train_seq) if len(sentence) == 1]
drop_test = [index for index, sentence in enumerate(decoder_input_test_seq) if len(sentence) == 1]

print('삭제할 훈련 데이터의 개수 :', len(drop_train))
print('삭제할 테스트 데이터의 개수 :', len(drop_test))

encoder_input_train_seq = [s for i, s in enumerate(encoder_input_train_seq) if i not in drop_train]
decoder_input_train_seq = [s for i, s in enumerate(decoder_input_train_seq) if i not in drop_train]
decoder_target_train_seq = [s for i, s in enumerate(decoder_target_train_seq) if i not in drop_train]
raw_text_train = [s for i, s in enumerate(raw_text_train) if i not in drop_train]

encoder_input_test_seq = [s for i, s in enumerate(encoder_input_test_seq) if i not in drop_test]
decoder_input_test_seq = [s for i, s in enumerate(decoder_input_test_seq) if i not in drop_test]
decoder_target_test_seq = [s for i, s in enumerate(decoder_target_test_seq) if i not in drop_test]
raw_text_test = [s for i, s in enumerate(raw_text_test) if i not in drop_test]

print('훈련 데이터의 개수 :', len(encoder_input_train_seq))
print('테스트 데이터의 개수 :', len(encoder_input_test_seq))

### 패딩하기

In [ ]:
def convert_to_tensor(sequences):
    return [torch.tensor(seq, dtype=torch.long) for seq in sequences]  # 정수 리스트 각각을 PyTorch 텐서로 변환

# 문장마다 길이(단어 수)가 다르므로, 짧은 문장 뒤에는 0(PAD)을 채우고 maxlen에 맞춰 자름
# → 그래야 여러 문장을 하나의 직사각형 모양 텐서(배치)로 묶어서 한 번에 계산할 수 있다
def pad_sequences_pytorch(sequences, maxlen, padding_value=0):
    sequences = convert_to_tensor(sequences)
    padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=padding_value)  # 배치 안에서 가장 긴 길이에 맞춰 0으로 채움
    return padded_seqs[:, :maxlen]  # maxlen보다 길면 그 이후는 잘라냄

encoder_input_train = pad_sequences_pytorch(encoder_input_train_seq, maxlen=text_max_len)
encoder_input_test = pad_sequences_pytorch(encoder_input_test_seq, maxlen=text_max_len)
decoder_input_train = pad_sequences_pytorch(decoder_input_train_seq, maxlen=summary_max_len)
decoder_target_train = pad_sequences_pytorch(decoder_target_train_seq, maxlen=summary_max_len)
decoder_input_test = pad_sequences_pytorch(decoder_input_test_seq, maxlen=summary_max_len)
decoder_target_test = pad_sequences_pytorch(decoder_target_test_seq, maxlen=summary_max_len)
print('=3')

## Step 3. 어텐션 메커니즘 사용하기 (추상적 요약)

인코더/디코더/어텐션/전체 모델 구조는 강의와 완전히 동일하다. 다만 이 데이터가 강의보다 크므로(샘플 수·단어장 크기 모두 큼) `num_layers`와 `batch_size`는 낮춰서, 먼저 파이프라인이 끝까지 도는지 확인하는 걸 우선으로 했다.

In [ ]:
embedding_dim = 128
hidden_size = 256
num_layers = 2   # 강의의 3에서 낮춤 — 데이터 규모가 커서 먼저 가볍게 확인 후 필요하면 올릴 것
dropout = 0.3

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=2, dropout=0.3):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)  # 단어 번호(정수) → 의미를 담은 벡터로 변환하는 층
        self.lstm = nn.LSTM(
            embedding_dim, hidden_size, num_layers=num_layers,
            dropout=dropout, batch_first=True
        )

    # forward: 이 클래스를 encoder(입력)처럼 함수처럼 호출하면 PyTorch가 자동으로 실행해주는 메서드
    def forward(self, x):
        embedded = self.embedding(x)  # (배치, 문장길이) → (배치, 문장길이, embedding_dim)
        output, (hidden, cell) = self.lstm(embedded)
        # output: 매 시점(단어)마다의 출력 전체 (→ 어텐션이 "원문 어디를 볼지" 계산할 때 씀)
        # hidden, cell: 문장을 다 읽은 후의 최종 상태 (→ 디코더의 시작점으로 넘겨줌, 이게 "문맥을 압축한 벡터")
        return output, hidden, cell

src_vocab_size = len(src_vocab)
encoder = Encoder(src_vocab_size, embedding_dim, hidden_size, num_layers=num_layers, dropout=dropout)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, dropout=0.3, num_layers=2):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_size, num_layers=num_layers, dropout=dropout,
            batch_first=True
        )

    def forward(self, x, hidden, cell):
        embedded = self.embedding(x)
        # (hidden, cell)을 초기 상태로 넘겨줌 → 인코더가 읽어낸 문맥에서 이어서 디코딩을 시작한다는 뜻
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        return output, hidden, cell

tar_vocab_size = len(tar_vocab)
decoder = Decoder(tar_vocab_size, embedding_dim, hidden_size, num_layers=num_layers, dropout=dropout)

In [ ]:
class Attention_dot(nn.Module):
    def __init__(self, hidden_size):
        super(Attention_dot, self).__init__()
        self.attn = nn.Linear(hidden_size, hidden_size)  # (이 dot-product 방식에서는 실제로 안 쓰이지만 클래스 인터페이스 통일용으로 남겨둠)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, decoder_output, encoder_outputs):
        # bmm(batch matrix multiply): 디코더가 "지금 이 시점"에서 인코더의 각 단어와 얼마나 관련있는지를
        # 내적(dot product)으로 계산 → 값이 클수록 그 원문 단어와 관련이 깊다는 뜻 (원시 점수, 아직 확률 아님)
        attn_weights = torch.bmm(decoder_output, encoder_outputs.transpose(1, 2))
        # softmax: 원시 점수들을 "합이 1인 확률(가중치)"로 바꿔줌 → 어느 단어에 얼마나 집중할지의 비율이 됨
        attn_weights = F.softmax(attn_weights, dim=-1)
        # 그 가중치로 encoder_outputs를 가중평균 → "지금 시점에 필요한 원문 정보만 뽑아낸" context 벡터
        attn_out = torch.bmm(attn_weights, encoder_outputs)
        return attn_out


class Seq2SeqWithAttention(nn.Module):
    def __init__(self, encoder, decoder, vocab_size, hidden_size):
        super(Seq2SeqWithAttention, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.attention = Attention_dot(hidden_size)
        self.concat = nn.Linear(hidden_size * 2, hidden_size)  # 디코더 출력 + 어텐션 context를 합친 걸 다시 hidden_size 크기로 압축
        self.output_layer = nn.Linear(hidden_size, vocab_size)  # 최종적으로 "단어장 크기만큼의 점수(logit)"로 변환 (다음 단어 예측용)

    def forward(self, encoder_input, decoder_input):
        encoder_outputs, hidden, cell = self.encoder(encoder_input)          # 원문을 읽고 문맥 요약(hidden/cell) + 시점별 출력 확보
        decoder_outputs, _, _ = self.decoder(decoder_input, hidden, cell)    # 그 문맥에서 시작해서 디코더가 출력을 생성

        attn_out = self.attention(decoder_outputs, encoder_outputs)  # 매 디코딩 시점마다 원문에서 관련 정보를 다시 끌어옴

        # cat(dim=-1): 디코더 자체 출력과 어텐션이 뽑아낸 정보를 이어붙임(concatenate) — 두 정보를 합쳐서 함께 판단하기 위함
        decoder_concat_output = torch.cat((decoder_outputs, attn_out), dim=-1)
        # tanh: 값을 -1~1 사이로 눌러주는 비선형 함수 — 이게 없으면 Linear층 두 개가 이어져도 결국 하나의 선형변환과 다를 게 없어짐
        decoder_concat_output = torch.tanh(self.concat(decoder_concat_output))
        output = self.output_layer(decoder_concat_output)  # 최종적으로 "다음 단어가 무엇일지"에 대한 단어장 크기의 점수(logit) 출력

        return output

model = Seq2SeqWithAttention(encoder, decoder, tar_vocab_size, hidden_size)
print(model)

### 모델 훈련하기

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

# Hyperparameters
batch_size = 128  # 강의의 256에서 낮춤 — 단어장이 커서 임베딩 층이 더 무거워진 것을 감안
epochs = 10  # 최대 반복 횟수(상한). patience=2인 EarlyStopping이 있어서, 검증 손실이 2번 연속 개선 안 되면 이 숫자에 도달하기 전에 먼저 멈춘다.
learning_rate = 0.001
patience = 2

criterion = nn.CrossEntropyLoss(ignore_index=0)  # 패딩 토큰 무시
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

train_dataset = TensorDataset(encoder_input_train, decoder_input_train, decoder_target_train)
test_dataset = TensorDataset(encoder_input_test, decoder_input_test, decoder_target_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
train_losses = []
val_losses = []

def train_model(model, train_loader, test_loader, criterion, optimizer, epochs, patience):
    model.train()
    best_val_loss = float('inf')
    early_stop_counter = 0

    for epoch in range(epochs):
        total_loss = 0

        for encoder_input, decoder_input, target in train_loader:
            optimizer.zero_grad()  # 이전 배치에서 계산된 기울기(gradient)가 남아있지 않도록 초기화

            encoder_input = encoder_input.to(device).long()
            decoder_input = decoder_input.to(device).long()
            target = target.to(device).long()

            output = model(encoder_input, decoder_input)
            # view(-1, ...): (배치, 문장길이, 단어장크기) 3차원을 (배치*문장길이, 단어장크기) 2차원으로 펼침
            # → CrossEntropyLoss는 "샘플 하나당 하나의 예측"을 가정하므로, 시점별 예측을 전부 한 줄씩 펼쳐서 넘겨줘야 함
            output = output.view(-1, output.shape[-1])
            target = target.view(-1)  # 정답도 같은 방식으로 1차원으로 펼침 (모양을 맞춰야 손실 계산 가능)

            loss = criterion(output, target)
            loss.backward()   # 손실(loss)을 기준으로 각 파라미터가 얼마나/어느 방향으로 바뀌어야 하는지 계산 (역전파)
            optimizer.step()   # 계산된 방향으로 실제 파라미터를 업데이트
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for encoder_input, decoder_input, target in test_loader:
                encoder_input = encoder_input.to(device).long()
                decoder_input = decoder_input.to(device).long()
                target = target.to(device).long()

                output = model(encoder_input, decoder_input)
                output = output.view(-1, output.shape[-1])
                target = target.view(-1)
                loss = criterion(output, target)

                val_loss += loss.item()

        val_loss /= len(test_loader)
        val_losses.append(val_loss)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f}")

        # EarlyStopping: 검증 손실이 이전 최고 기록보다 좋아졌으면 카운터 초기화,
        # 나빠지거나 그대로면 카운터 증가 → patience번 연속으로 개선이 없으면 학습을 중단
        # (계속 돌려도 성능이 안 느는데 시간만 쓰는 것을 막고, 과적합 직전에서 멈추기 위함)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stop_counter = 0
        else:
            early_stop_counter += 1

        if early_stop_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

        model.train()  # 검증(eval) 동안 꺼뒀던 dropout 등을 다시 학습 모드로 되돌림

In [ ]:
%%time
train_model(model, train_loader, test_loader, criterion, optimizer, epochs=epochs, patience=patience)

루브릭 2번 항목(학습 성공 확인)에 필요한 loss 그래프 — train/validation loss가 감소하는 경향을 보여야 한다.

In [ ]:
plt.plot(range(len(train_losses)), train_losses, label='Train Loss')
plt.plot(range(len(val_losses)), val_losses, label='Validation Loss')
plt.legend()
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.show()

## Step 4. 실제 결과와 요약문 비교하기 (추상적 요약)

In [ ]:
# vocab은 "단어 → 번호"인데, 예측 결과(번호)를 다시 사람이 읽을 단어로 바꾸려면 반대 방향 사전이 필요함
src_index_to_word = {idx: word for word, idx in src_vocab.items()}  # 원문 단어장: 번호 → 단어
tar_word_to_index = tar_vocab                                        # 요약 단어장: 단어 → 번호 (이름만 새로 붙임)
tar_index_to_word = {idx: word for word, idx in tar_vocab.items()}   # 요약 단어장: 번호 → 단어

print('=3')

In [ ]:
# 학습이 끝난 모델로 실제 요약을 "생성"하는 함수 — 훈련 때와 달리 정답을 모르는 상태에서
# 한 단어씩 예측하고, 그 예측을 다음 입력으로 다시 넣는 자기회귀(autoregressive) 방식으로 동작
def decode_sequence(input_seq, model, tar_word_to_index, tar_index_to_word, summary_max_len, device):
    model.eval()  # dropout 등을 끄고 평가 모드로 전환 (훈련 때와 동일하게 항상 같은 결과가 나오도록)
    input_seq = input_seq.to(device)
    decoded = []
    with torch.no_grad():  # 예측만 할 뿐 학습은 안 하므로, 기울기 계산을 꺼서 메모리/속도를 아낌
        e_out, h, c = model.encoder(input_seq)  # 원문을 인코더에 통과시켜 문맥(h, c)과 시점별 출력을 확보
        # sostoken(시작 토큰)으로 디코딩을 시작 — (1,1) 모양: 배치 1개, 시점 1개짜리 입력
        target_seq = torch.tensor([[tar_word_to_index['sostoken']]], device=device)
        for _ in range(summary_max_len - 1):
            dec_out, h, c = model.decoder(target_seq, h, c)          # 현재 입력 한 단어로 디코더를 한 스텝 진행
            attn_out = model.attention(dec_out, e_out)               # 원문에서 지금 필요한 정보를 다시 끌어옴
            concat = torch.tanh(model.concat(torch.cat((dec_out, attn_out), dim=-1)))
            logits = model.output_layer(concat)                      # 단어장 크기만큼의 점수
            idx = logits[0, -1].argmax().item()                      # argmax: 점수가 가장 높은(=가장 그럴듯한) 단어 번호 선택
            token = tar_index_to_word[idx]
            if token == 'eostoken':  # 종료 토큰이 나오면 문장이 끝났다는 뜻이므로 멈춤
                break
            decoded.append(token)
            target_seq = torch.tensor([[idx]], device=device)  # 방금 예측한 단어를 다음 스텝의 입력으로 사용 (자기회귀)
    return ' '.join(decoded)

In [ ]:
# 원문의 정수 시퀀스를 텍스트 시퀀스로 변환
def seq2text(input_seq):
    temp = ''
    for i in input_seq:
        key = int(i.item())
        if key != 0:
            temp = temp + src_index_to_word.get(key, "<UNK>") + ' '
    return temp.strip()

# 요약문의 정수 시퀀스를 텍스트 시퀀스로 변환
def seq2summary(input_seq):
    temp = ''
    for i in input_seq:
        key = int(i.item())
        if key != 0 and key != tar_word_to_index['sostoken'] and key != tar_word_to_index['eostoken']:
            temp = temp + tar_index_to_word.get(key, "<UNK>") + ' '
    return temp.strip()

print('=3')

In [ ]:
# 실제 요약(headlines)과 예측 요약을 비교하고, 핵심 단어가 얼마나 겹치는지도 함께 확인한다.
comparison_records = []  # Step 5에서 추출적 요약 결과를 이어붙여 비교표를 만들 때 재사용

for i in range(50, 100):
    original_text = seq2text(encoder_input_test[i])
    actual_summary = seq2summary(decoder_input_test[i])
    input_seq = encoder_input_test[i].unsqueeze(0)  # (T,) -> (1, T), 배치 차원 추가
    predicted_summary = decode_sequence(input_seq, model, tar_word_to_index, tar_index_to_word, summary_max_len, device)

    actual_words = set(actual_summary.split())
    predicted_words = set(predicted_summary.split())
    overlap = actual_words & predicted_words
    overlap_ratio = len(overlap) / len(actual_words) if len(actual_words) > 0 else 0

    print("원문 :", original_text)
    print("실제 요약 :", actual_summary)
    print("예측 요약 :", predicted_summary)
    print(f"겹치는 핵심 단어 ({len(overlap)}/{len(actual_words)}, {overlap_ratio:.0%}) :", overlap)
    print("\n")

    comparison_records.append({
        "raw_text": raw_text_test[i],
        "actual": actual_summary,
        "predicted_abstractive": predicted_summary,
        "keyword_overlap_ratio": overlap_ratio,
    })

avg_overlap = sum(r["keyword_overlap_ratio"] for r in comparison_records) / len(comparison_records)
print(f"평균 핵심 단어 겹침 비율: {avg_overlap:.1%}")

## Step 5. Summa를 이용해서 추출적 요약해보기

Summa는 문장 토큰화를 내부적으로 처리하는데, 이건 마침표·대문자 같은 문장 경계가 남아있어야 가능하다. Step 2에서 정제한 `text`는 그런 경계를 이미 다 지웠으므로, 여기서는 Step 1에서 미리 보존해둔 **원본** `raw_text_test`를 사용한다.

In [ ]:
from summa.summarizer import summarize

for record in comparison_records:
    extractive_summary = summarize(record["raw_text"], ratio=0.4)
    # ratio=0.4: 뉴스 기사 한 편은 문장 수가 적어서, 강의의 0.01(매트릭스 시놉시스 전체)보다
    # 훨씬 큰 비율을 줘야 결과가 나온다. 너무 짧거나 길면 이 값을 조정할 것.
    if extractive_summary.strip() == '':
        extractive_summary = summarize(record["raw_text"], words=15)

    record["extractive_summary"] = extractive_summary

    actual_words = set(record["actual"].split())
    extractive_words = set(extractive_summary.lower().split())
    overlap_ext = actual_words & extractive_words
    record["extractive_keyword_overlap_ratio"] = (
        len(overlap_ext) / len(actual_words) if len(actual_words) > 0 else 0
    )

print('=3')

In [ ]:
comparison_df = pd.DataFrame(comparison_records)
comparison_df = comparison_df[[
    "actual", "predicted_abstractive", "extractive_summary",
    "keyword_overlap_ratio", "extractive_keyword_overlap_ratio"
]]
comparison_df.columns = [
    "실제 요약(headlines)", "추상적 요약(예측)", "추출적 요약(Summa)",
    "추상적_핵심단어겹침비율", "추출적_핵심단어겹침비율"
]

print("추상적 요약 평균 핵심단어 겹침 비율:", comparison_df["추상적_핵심단어겹침비율"].mean())
print("추출적 요약 평균 핵심단어 겹침 비율:", comparison_df["추출적_핵심단어겹침비율"].mean())

comparison_df.to_csv("abstractive_vs_extractive_comparison.csv", index=False, encoding='utf-8-sig')
comparison_df.head(10)

**문법완성도 비교 (직접 10개 정도 읽어보고 채워 넣을 것 — 자동 채점 불가능한 부분)**

- 추상적 요약: (예: 문장 구조는 완전하지만 원문에 없는 단어를 만들어내거나 핵심 인물/숫자를 놓치는 경우가 있었다 등)
- 추출적 요약: (예: 원문 문장을 그대로 가져오니 문법은 항상 자연스럽지만, 문장을 이어붙이면 맥락 연결이 부자연스러운 경우가 있었다 등)

## 회고 (초안 — 실제 겪은 대로 고쳐서 써야 함)

아래는 이번 프로젝트에서 실제로 판단이 필요했던 지점들을 바탕으로 뼈대만 잡아본 것이다. 실제로 코드를 돌려보면서 겪은 것에 맞게 직접 고쳐서 쓰는 게 좋다.

**배운 점**
- 강의에서 배운 파이프라인을 그대로 복사하는 게 아니라, 데이터가 바뀌면 무엇을(불용어 제거 여부, 최대 길이, 단어 집합 크기) 다시 판단해야 하는지 감을 잡았다.
- Summa 같은 추출적 요약 도구는 문장 경계(마침표·대문자)에 의존하기 때문에, 추상적 요약용으로 정제한 텍스트를 그대로 재사용하면 안 된다는 걸 배웠다 — 정제 파이프라인을 설계할 때 "이 정제된 데이터를 나중에 또 어디에 쓸지"까지 미리 생각해야 한다는 교훈.

**아쉬운 점**
- (실제로 돌려보고 느낀 아쉬운 점을 적을 것 — 예: 예측 요약의 핵심단어 겹침 비율이 예상보다 낮았다든지, 학습 시간이 오래 걸려서 하이퍼파라미터를 여러 번 실험해보기 어려웠다든지)

**느낀 점**
- (실제로 결과를 보고 느낀 점을 적을 것 — 예: 추출적 요약과 추상적 요약 중 어느 쪽이 이 데이터에 더 잘 맞았는지, 그 이유를 스스로 추측해보는 것도 좋음)

---

### 디버깅 기록 / 추가 실험 (직접 채워야 하는 부분)

코드를 실제로 돌리다가 에러가 나거나 예상과 다르게 동작한 부분, 혹은 하이퍼파라미터(`num_layers`, `batch_size`, `ratio` 등)를 바꿔서 실험해본 게 있다면 여기에 기록한다.